# Linear Regression

## Objectives
The objective of this notebook is to prepare the cleaned data for a linear regression. Sections 1 to 5 summarise the theory of the model, because it shapes the decisions made along the way: the model and its solution, the assumptions, bias and variance, the Gauss–Markov theorem, hypothesis testing and goodness of fit. Feature engineering then uses that theory to make the data suit the assumptions, and the target is chosen once a model has been fitted. Here $n$ is the number of houses, $k = p + 1$ is the number of coefficients ($p$ features and an intercept), and $X$ is the $n \times k$ matrix whose first column is ones.

## 1. The Model
The model is
$$y = X\beta + \varepsilon$$
- $\beta$ are fixed, unknown population coefficients.
- $\varepsilon$ is everything the features do not explain.

### Solution
Ordinary least squares chooses $b$ to minimise the sum of squared residuals $S(b) = (y - Xb)^\top (y - Xb)$. Setting the gradient to zero gives the normal equations:
$$\frac{\partial S}{\partial b} = -2X^\top y + 2X^\top X b = 0 \;\Rightarrow\; X^\top X \hat\beta = X^\top y$$
If the columns of $X$ are linearly independent, $X^\top X$ is invertible and
$$\hat\beta = (X^\top X)^{-1} X^\top y$$
- The second derivative $2X^\top X$ is positive definite, so this is the unique minimum.
- Geometrically, $\hat y = X\hat\beta$ is the projection of $y$ onto the column space of $X$. The residuals $e = y - X\hat\beta$ are orthogonal to every column, so with an intercept they sum to zero.
- Computing $\hat\beta$ needs no assumptions. The assumptions are about what $\hat\beta$ tells us about $\beta$.
- Without full rank, $\hat\beta$ is not unique, but the fitted values are.

## 2. Assumptions
The properties of $\hat\beta$ below rest on these assumptions about $X$ and $\varepsilon$:
- The columns of $X$ are linearly independent (full column rank, $n \ge k$), so $\hat\beta$ is unique.
- The relationship is linear, $E[y \mid X] = X\beta$, which means the model is correctly specified.
- The errors are exogenous, $E[\varepsilon \mid X] = 0$. They are unrelated to the features, and with an intercept any constant mean is absorbed.
- The errors are homoskedastic, $\text{Var}(\varepsilon_i \mid X) = \sigma^2$. The variance is constant and does not depend on the features.
- The errors are uncorrelated, $\text{Cov}(\varepsilon_i, \varepsilon_j \mid X) = 0$ for $i \ne j$. Together with the previous point, $\text{Var}(\varepsilon \mid X) = \sigma^2 I$.
- The errors are normal, $\varepsilon \sim N(0, \sigma^2 I)$. This is the only optional assumption, and it is needed only for exact small-sample inference.

## 3. Bias, Variance and the Gauss–Markov Theorem
Substituting $y = X\beta + \varepsilon$ gives
$$\hat\beta = \beta + (X^\top X)^{-1} X^\top \varepsilon$$
The error in $\hat\beta$ is a weighted sum of the noise. A different sample of houses has different noise, so $\hat\beta$ is random, and its properties follow from how $\varepsilon$ behaves.

### Bias
$$E[\hat\beta \mid X] - \beta = (X^\top X)^{-1} X^\top E[\varepsilon \mid X]$$
- This is zero under exogeneity. If part of the noise moves with a feature, OLS credits that feature with an effect that belongs to the noise.
- Fitting a line to a curve leaves the curve in $\varepsilon$, so misspecification is the same problem. It biases the predictions as well as the coefficients.

### Variance
$$\text{Var}(\hat\beta \mid X) = (X^\top X)^{-1} X^\top \,\text{Var}(\varepsilon \mid X)\, X (X^\top X)^{-1} = \sigma^2 (X^\top X)^{-1} \quad \text{if } \text{Var}(\varepsilon \mid X) = \sigma^2 I$$
- It grows with the noise $\sigma^2$, and shrinks with more rows and more spread in the features.
- It grows when columns are nearly collinear, which is what the variance inflation factor (VIF) measures.

### Gauss–Markov Theorem
- Under linearity, exogeneity, full rank, constant variance and uncorrelated errors, OLS is the best linear unbiased estimator (BLUE).
- For any other linear unbiased estimator $\tilde\beta = Cy$, the matrix $\text{Var}(\tilde\beta) - \text{Var}(\hat\beta)$ is positive semidefinite, so every combination $c^\top \hat\beta$ has the smallest variance.
- Normality is not needed. Any other such estimator is OLS plus a term that is invisible to the features, which adds only variance.

##### Implications:
- Best is only among unbiased estimators. If the variances differ or the errors are correlated, OLS is still unbiased but not best, and its standard errors are wrong. Weighted or generalised least squares weights each house by the inverse of its error variance.
- Ridge and lasso are biased, so they fall outside the theorem and can have lower total error. This is the bias–variance trade-off.
- The theorem says nothing about whether the model is right. Linearity and exogeneity decide the bias.
- The variance of a prediction at a new house $x_0$ is $\sigma^2 + \sigma^2 x_0^\top (X^\top X)^{-1} x_0$, which is larger for houses unlike the training data.

## 4. Hypothesis Testing
Under normal errors, $\hat\beta \sim N(\beta, \sigma^2 (X^\top X)^{-1})$, and $\sigma^2$ is estimated by $\hat\sigma^2 = e^\top e / (n - k)$.

### t-test
This tests one coefficient:
$$t_j = \frac{\hat\beta_j - \beta_{j0}}{\text{se}(\hat\beta_j)}, \qquad \text{se}(\hat\beta_j) = \hat\sigma \sqrt{[(X^\top X)^{-1}]_{jj}} \;\sim\; t_{n-k} \text{ under } H_0$$
- The usual null is $H_0: \beta_j = 0$. It asks whether the feature adds anything given the other features in the model.
- A confidence interval is $\hat\beta_j \pm t_{n-k,\,1-\alpha/2}\, \text{se}(\hat\beta_j)$.

### F-test
This tests $q$ restrictions on the coefficients:
$$F = \frac{(\text{RSS}_r - \text{RSS}_u)/q}{\text{RSS}_u/(n-k)} \;\sim\; F_{q,\,n-k} \text{ under } H_0$$
- $\text{RSS}_r$ is from the restricted model and $\text{RSS}_u$ from the full model. It tests whether a group of coefficients, such as all the dummies of one category, is jointly zero.
- With all $p$ slopes restricted to zero it is the overall test, and with $q = 1$ it equals $t^2$.

##### When the tests can be trusted:
- Without normality they are approximately valid for large $n$, by the central limit theorem.
- Without constant variance or uncorrelated errors the standard errors are wrong, so the tests and intervals are wrong. Robust standard errors are one fix.
- Ridge and lasso give biased coefficients, so these tests no longer apply.

## 5. Goodness of Fit
The coefficient of determination and its adjusted version are
$$R^2 = 1 - \frac{\text{RSS}}{\text{TSS}}, \qquad \bar R^2 = 1 - \frac{\text{RSS}/(n-k)}{\text{TSS}/(n-1)}$$
- $R^2$ is the share of the variance of $y$ that the model explains. It needs an intercept to be defined this way.
- It never falls when a column is added, even a useless one. Adjusted $\bar R^2$ penalises the extra coefficients.
- It is measured on the training data, so it says nothing about prediction on new houses. A high $R^2$ is also compatible with a misspecified model.
- For prediction we care about the error on houses the model has not seen, which is the competition score on the test set.

## 6. Feature Engineering
The theory above says what the features must satisfy. The columns of $X$ must be linearly independent for $\hat\beta$ to exist (section 1), the relevant drivers of price must be present for the errors to be exogenous, and the columns must be in a form where the relationship is linear (section 2). This section makes the data numeric and the columns independent, using the decisions from the EDA. It leaves the target out, because the target is chosen once a model has been fitted. Everything is in `src/engineering.py`, and `engineer` applies it to train and test with the levels taken from train only, so no test information is used. We ask:
- Which columns are replaced by a combination, from the EDA decisions?
- How are the categories turned into numbers without creating dependent columns?
- Are the resulting columns linearly independent, and are any nearly so?
- Which transformations need a fitted model, and so wait?

### Combining and Encoding
The EDA found columns that repeat or sum others. These are replaced by combinations:
- `TotalBathrooms` replaces the four bathroom columns, with each half bath counting 0.5.
- `HouseAge`, `RemodelAge` and `GarageAge` (the year sold minus the year built, remodelled and the garage built) replace the three year columns.
- `HasSecondExterior` replaces `Exterior2nd`, and is 1 when it differs from `Exterior1st` once the spelling is matched.
- `1stFlrSF` and `BsmtUnfSF` are dropped, because each total (`GrLivArea` and `TotalBsmtSF`) is the exact sum of its parts. `HasGarage` is dropped because it equals the `None` level of `GarageType`.

The categories are encoded in two ways:
- The columns with a natural order, or two levels, are coded as integers. This is the quality scales (`None` to `Ex` as 0 to 5), basement exposure and finish, `Functional`, `GarageFinish`, `PavedDrive`, `LotShape`, `LandSlope`, `CentralAir` and `Street`.
- The others are one-hot encoded, including the numeric codes `MSSubClass` and `MoSold`, which are categories. The most common level of each is the reference and gets no column, so a set of dummies cannot add up to the intercept. A level with fewer than 10 train houses, or a level not seen in train, is grouped as `Other`.

The minimum of 10 is a choice, so we look at how many columns each value gives.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.linear_model import LinearRegression

from src.data import load_train, load_test
from src.engineering import engineer
from src.features import clean

train, test = load_train(), load_test()
train_clean, test_clean = clean(train, train), clean(test, train)

# Columns for each minimum number of train houses per level.
widths = {count: engineer(train_clean, train_clean, count, drop_duplicates=False).shape[1] for count in [1, 5, 10, 20, 30]}
display(pd.DataFrame({"Columns": widths}).rename_axis("Minimum houses per level").T)

# Encode train and test, with the levels from train only.
features = engineer(train_clean, train_clean, drop_duplicates=False)
test_features = engineer(test_clean, train_clean, drop_duplicates=False)
print("Shapes:", features.shape, test_features.shape)
print("Same columns in train and test:", list(features.columns) == list(test_features.columns))
print("NaNs remaining (train, test):", features.isnull().sum().sum(), test_features.isnull().sum().sum())

# The combined columns.
combined = ["TotalBathrooms", "HouseAge", "RemodelAge", "GarageAge", "HasSecondExterior"]
display(features[combined].describe().loc[["min", "mean", "max"]].round(2))
print("Test houses with an MSSubClass not in train:", (~test_clean["MSSubClass"].isin(train_clean["MSSubClass"])).sum())

##### Findings:
- The number of columns falls as the minimum rises, from 209 columns with no grouping to 178 at 10 houses and 149 at 30. At 10, the 1,458 train houses have 178 columns (one is dropped below), so there are many more rows than columns.
- Train and test have the same columns and no missing values.
- The combined columns have sensible ranges.
- One test house has an `MSSubClass` (150) that does not appear in train, and is grouped as `Other`.

##### Decision:
Use a minimum of 10 houses per level. Rare levels would give columns that are nonzero for a handful of houses, and each such house has a lot of influence on its own coefficient.

##### Note on the ordered columns:
- The ordered columns are coded as integers because a regression needs numbers. The order comes from the data description.
- The description gives the order of the levels but not the size of the steps, apart from `BsmtQual`, whose levels are 10-inch bands of height. Integers assume equal steps (1, 2, 3, 4, 5), so where the true steps differ, the coding is wrong for that column.
- The weakest cases are `BsmtFinType1` and `BsmtFinType2`, which mix finish type and quality, and `Functional`, where almost every house is `Typ`. `NA` is also coded as 0 although it means no feature and not poor quality, which matters little because other columns carry the presence.
- This is revisited with a model, by comparing the integers with dummies for each column.

### Linear Independence
The columns are independent if the design matrix, with its intercept column, has a rank equal to its number of columns. If it is short, some column is an exact combination of others, and we look for it as in the EDA: standardise the columns and find the zero singular values. We then look for near dependence with the variance inflation factor, $\text{VIF}_j = 1 / (1 - R_j^2)$, where $R_j^2$ is from regressing column $j$ on the others. A value above 10 is the usual warning sign. We ask:
- Is the design matrix full rank?
- If not, which columns are the exact dependency?
- Which columns are nearly dependent on others?

In [ ]:
# Rank of the design matrix, including the intercept.
design = np.column_stack([np.ones(len(features)), features.to_numpy(dtype=float)])
print(f"{design.shape[1]} columns, rank {np.linalg.matrix_rank(design)}")

# Exact dependencies: singular values of zero in the standardised columns.
standardised = (features - features.mean()) / features.std(ddof=0)
_, singular_values, directions = np.linalg.svd(standardised.to_numpy(), full_matrices=False)
for direction in directions[singular_values < 1e-8 * singular_values[0]]:
    weights = pd.Series(direction, index=features.columns)
    print(weights[weights.abs() > 0.05].round(2).to_dict())
print("MSSubClass_90 equals BldgType_Duplex:", (features["MSSubClass_90"] == features["BldgType_Duplex"]).all())

# Drop the duplicate column and check again.
features = engineer(train_clean, train_clean)
test_features = engineer(test_clean, train_clean)
design = np.column_stack([np.ones(len(features)), features.to_numpy(dtype=float)])
print(f"\nAfter dropping BldgType_Duplex: {design.shape[1]} columns, rank {np.linalg.matrix_rank(design)}")

# Variance inflation factors, for near dependence: the diagonal of the inverse correlation matrix.
vif = pd.Series(np.diag(np.linalg.inv(np.corrcoef(features, rowvar=False))), index=features.columns).sort_values(ascending=False)
print(f"Columns with a VIF above 10: {(vif > 10).sum()}, above 5: {(vif > 5).sum()}, above 100: {(vif > 100).sum()}\n")
display(vif[vif > 10].round(1).rename("VIF").to_frame())

##### Findings:
- The matrix is one column short of full rank: 179 columns (178 plus the intercept) and a rank of 178. The exact dependency is two dummy columns, `MSSubClass_90` and `BldgType_Duplex`, which are identical. Every house with an `MSSubClass` of 90 is a duplex, and every duplex has an `MSSubClass` of 90.
- This dependency is between two different categorical columns, so leaving out a reference level for each category does not remove it. Only a rank check finds it.
- No column has a VIF above 100, but 22 are above 10 and 41 above 5.
- The columns with the highest VIFs overlap in meaning. They are the dwelling type and style columns (`MSSubClass`, `BldgType` and `HouseStyle`), `SaleType_New` with `SaleCondition_Partial`, `2ndFlrSF` with `GrLivArea` and `HouseStyle_2Story`, and `GarageQual` with `GarageCond`.

##### Decision:
Drop `BldgType_Duplex`, which gives a full-rank matrix of 177 columns plus the intercept. Both columns carry the same information, and `MSSubClass_90` is kept.

Do not drop the high-VIF columns yet. A high VIF does not make the matrix singular and has little effect on the fitted values, but it inflates the variance of the individual coefficients (section 3), so those coefficients cannot be read one by one. The goal is prediction, so they stay in for now, and ridge or lasso could handle them later if they hurt.

## 7. Target
The competition scores the root mean squared error between the log of the predicted price and the log of the actual price. A least-squares regression minimises squared error on the scale of its target, so fitting $\log(\text{SalePrice})$ minimises the quantity that is scored. This fixes the target. The assumptions in section 2 are about the errors, so they are checked on the residuals of the fitted model, in the next sections.

In [ ]:
# The price and the log price.
price = train_clean["SalePrice"]
target = np.log(price)

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
for ax, (name, values) in zip(axes, {"Price": price, "Log price": target}.items()):
    ax.hist(values, bins=40)
    ax.set_title(f"{name} (skew {stats.skew(values):.2f})")
plt.tight_layout()
plt.show()

##### Decision:
Use $\log(\text{SalePrice})$ as the target. It is the quantity the competition scores, and the plot shows it is also much more symmetric than the price.

Notes:
- The target is chosen by the metric, not by the assumptions. If the residuals of the fitted model break an assumption, we change the features, not the target.
- A prediction is turned back into a price with the exponential. The competition takes the log of the price again, so no further correction is needed for the score.

## 8. Baseline Model
We fit the regression by hand with numpy, so that each quantity is the formula from the theory above. The model is $\log(\text{SalePrice})$ on the engineered features of section 6 and an intercept, fitted to all the train houses. We ask:
- Does the hand-written fit match a library?
- How much of the variance does the model explain, and is it useful overall?
- Which coefficients differ from zero, and how far can we trust them one by one?

In [ ]:
# Design matrix with an intercept, and the log target.
X = np.column_stack([np.ones(len(features)), features.to_numpy(dtype=float)])
y = target.to_numpy()
n, k = X.shape

# Least squares. lstsq is the stable form of the normal equations.
beta_hat = np.linalg.lstsq(X, y, rcond=None)[0]
residuals = y - X @ beta_hat

# Check against the normal equations and scikit-learn.
print(f"Largest difference from the normal equations: {np.abs(beta_hat - np.linalg.solve(X.T @ X, X.T @ y)).max():.1e}")
library = LinearRegression().fit(features, y)
print(f"Largest difference from scikit-learn: {max(np.abs(beta_hat[1:] - library.coef_).max(), abs(beta_hat[0] - library.intercept_)):.1e}")
print(f"Residuals are orthogonal to every column: {np.abs(X.T @ residuals).max():.1e}, and sum to {residuals.sum():.1e}")

# Fit statistics.
rss = residuals @ residuals
tss = ((y - y.mean()) ** 2).sum()
f_statistic = ((tss - rss) / (k - 1)) / (rss / (n - k))
fit = {
    "Houses": n,
    "Coefficients": k,
    "Residual standard error": np.sqrt(rss / (n - k)),
    "R-squared": 1 - rss / tss,
    "Adjusted R-squared": 1 - (rss / (n - k)) / (tss / (n - 1)),
    "F statistic": f_statistic,
    "F p-value": stats.f.sf(f_statistic, k - 1, n - k),
    "RMSE of log price (in sample)": np.sqrt(rss / n),
}
display(pd.Series(fit, name="Baseline").to_frame())

In [ ]:
# Standard errors: sigma squared times the diagonal of (X'X)^-1, which the SVD of X gives as V S^-2 V'.
sigma_squared = rss / (n - k)
_, singular_values, right_vectors = np.linalg.svd(X, full_matrices=False)
standard_errors = np.sqrt(sigma_squared * np.sum((right_vectors.T / singular_values) ** 2, axis=1))
print(f"Largest relative difference from the explicit inverse: {np.max(np.abs(standard_errors - np.sqrt(sigma_squared * np.diag(np.linalg.inv(X.T @ X)))) / standard_errors):.1e}")

# t-tests of each coefficient against zero.
t_values = beta_hat / standard_errors
coefficients = pd.DataFrame(
    {"Coefficient": beta_hat, "Standard error": standard_errors, "t": t_values, "p-value": 2 * stats.t.sf(np.abs(t_values), n - k)},
    index=["Intercept", *features.columns],
)
print(f"Coefficients with a p-value below 0.05: {(coefficients['p-value'] < 0.05).sum()} of {k}, below 0.001: {(coefficients['p-value'] < 0.001).sum()}")
display(coefficients.reindex(coefficients["t"].abs().sort_values(ascending=False).index).head(15).round(6))

# Standard error against the variance inflation factor, for the columns with a high VIF.
high = vif[vif > 10].index
print(f"Median |t| for the {len(high)} columns with a VIF above 10: {coefficients.loc[high, 't'].abs().median():.2f}, and for the others: {coefficients.drop(index=['Intercept', *high])['t'].abs().median():.2f}")

##### Findings:
- The hand-written fit matches the library. The coefficients agree with the normal equations to 3e-9 and with scikit-learn to 6e-11, and the residuals are orthogonal to every column and sum to zero, as section 1 says.
- The model explains 94.3% of the variance of the log price, or 93.5% adjusted. The F statistic of 118.9 has a p-value of effectively 0, so the features are useful together.
- The residual standard error is 0.102, which is a typical error of about 10% in the price. The in-sample RMSE of the log price is 0.096. It is measured on the houses used for the fit, with 178 coefficients for 1,458 houses, so it is optimistic. The error on unseen houses is the competition score, which we get by submitting.
- 45 of the 178 coefficients differ from zero at the 5% level, and 21 at the 0.1% level. The largest t-statistics are for `GrLivArea` (11.8), `OverallCond` (9.9), `MSZoning_C (all)` (-9.5) and `OverallQual` (9.4).
- The coefficients are proportional changes in price. Each extra square foot of `GrLivArea` adds about 0.027% (100 square feet is about 2.7%), each point of `OverallQual` about 4.2%, and each year of `HouseAge` takes off about 0.18%. A `MSZoning_C (all)` house sells for about 35% less, as exp(-0.424) is 0.65.
- The 22 columns with a VIF above 10 have a lower median absolute t-statistic than the rest (0.73 against 1.17). This is the variance inflation from section 3: their standard errors are larger, so they are harder to tell from zero one by one.

##### Decision:
Keep this model as the baseline. Any later change is judged by the diagnostics and by the competition score after submitting. The coefficients of the high-VIF columns are not read one by one, because the goal is prediction. The next section checks the residuals of this model against the assumptions.

## 9. Diagnostics
The assumptions in section 2 are about the errors, and the residuals $e = y - X\hat\beta$ of the fit above, the actual log price minus the fitted one, are our estimate of them. We check them with two plots:
- Residuals against fitted values. If the model is right, the residuals scatter around 0 with the same vertical spread everywhere. A curve means the line is missing a shape, which is the linearity assumption. A funnel means the spread changes, which is the constant variance assumption. The red line is the mean residual in each tenth of the fitted values.
- A normal Q-Q plot. The sorted residuals are plotted against the values a normal distribution would give at the same ranks. Points on the diagonal mean the residuals are normal, and tails that bend away mean heavy tails. Normality matters for the exact t and F tests, and for prediction intervals. We ask:
- Is a curve left in the residuals?
- Does their spread change with the fitted value?
- Are they close to normal, or are the tails heavy?

In [ ]:
def binned_mean(x, values, bins=10):
    # Mean of x and of values in each bin of x, to show a trend.
    frame = pd.DataFrame({"x": x, "v": values}).groupby(pd.qcut(x, bins), observed=True).mean()
    return frame["x"], frame["v"]


fitted = X @ beta_hat
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].scatter(fitted, residuals, s=6, alpha=0.4)
axes[0].plot(*binned_mean(fitted, residuals), color="red")
axes[0].axhline(0, color="black", linewidth=1)
axes[0].set(title="Residuals against fitted values", xlabel="Fitted log price", ylabel="Residual")
stats.probplot(residuals, plot=axes[1])
axes[1].set_title("Normal Q-Q")
plt.tight_layout()
plt.show()

# Numbers behind the plots: the spread by fifth of the fitted values, and the residuals far from 0.
quintile = pd.qcut(fitted, 5, labels=range(1, 6))
display(pd.Series(residuals).groupby(quintile, observed=True).std().round(3).rename("Residual standard deviation").rename_axis("Fitted quintile").to_frame().T)
far = (np.abs(residuals) > 3 * np.sqrt(sigma_squared)).sum()
print(f"Residuals more than 3 standard errors from 0: {far}, against about {n * 2 * stats.norm.sf(3):.0f} for a normal distribution")

##### Findings:
- No curve is left. The mean residual, the red line, stays close to 0 across the fitted values.
- The spread is not constant. It is widest at the low end: the residual standard deviation is 0.133 for the lowest fifth of the fitted values, against 0.076 to 0.094 for the rest. It does not widen with price in the way that the price itself would.
- The tails are heavy. The middle of the Q-Q plot follows the line, but both tails bend away, the lower one more. 18 residuals are more than 3 standard errors from 0, against about 4 for a normal distribution. The most negative are around -0.6 to -0.7, which is a house that sold for about half the predicted price.